## 1. 머신러닝 모델 하이퍼 파라미터 튜닝

In [ ]:
'''
하이퍼 파라미터란 모델 학습시에 사용자가 사전에 설정하는 값을 의미한다.
하이퍼 파라미터의 설정에 따라 모델의 성능이 좌우된다.

하이퍼 파라미터를 잘못 설정한 경우
    Underfitting
    모델이 학습을 충분히 하지 못한 상황
    Overfitting
    모델이 학습 데이터에 과하게 fit되어 일반화가 안되는 상황
    

'''

### 1-1. 회귀 모델 하이퍼 파라미터

In [ ]:
# 선형회귀 하이퍼 파라미터

'''
L1 정규화
모델의 가중치가 과도하게 커지는 것을 막아 과적합 예방

장점
변수를 선택, 제거할 수 있어 희소한 모델을 생성할 수 있고 선형 모델과 잘 
Lasso Regression



L2 정규화
모델의 가중치가 너무 커지지 않게 막아 과적합 예방

장점 
모든 변수의 계수를 조금씩 줄이는 방식으로 모든 변수를 사용함
변수를 제거하지 않아 모델이 비교적 안정적임 - 해석보다는 예측 성능 중심


ElasticNet
L1과 L2정규화를 결합한 형태
필요없는 변수는 제거하고(L1) 나머지 변수들은 균형있게 조정(L2)
시간이 부족한 상황이나 시작 단계에서 사용하고
이후 상황에 따라 L1, L2, ElasticNet 가운데 선택하면 된다.

    L1      - 변수 중 일부만이 의미 있는 경우 (희소한 해석이 필요)
    L2      - 모든 변수가 조금씩 중요할 때
    Elastic - 변수의 수가 많고 변수들 간의 상관관계가 높은 경우
'''

##### 선형회귀

In [ ]:
import pandas as pd
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 데이터 로드 및 전처리
df = pd.read_csv('datasets/Clean_Dataset.csv')
df = df.drop(['flight', 'departure_time', 'stops', 'arrival_time'], axis=1) # 학습에 필요 없는 문자열 열 제거
df = pd.get_dummies(df, columns=['airline', 'source_city', 'destination_city', 'class'], drop_first=True) # 원 핫 인코딩
X = df.drop('price', axis=1)  # 독립 변수
y = df['price']  # 종속 변수

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ridge Regression (L2 정규화)
ridge = Ridge(alpha=1.0)  # L2 정규화 강도
ridge.fit(X_train, y_train)
ridge_preds = ridge.predict(X_test)
ridge_mse = mean_squared_error(y_test, ridge_preds)

# Lasso Regression (L1 정규화)
lasso = Lasso(alpha=0.1)  # L1 정규화 강도
lasso.fit(X_train, y_train)
lasso_preds = lasso.predict(X_test)
lasso_mse = mean_squared_error(y_test, lasso_preds)

print("Ridge Regression MSE:", ridge_mse)
print("Lasso Regression MSE:", lasso_mse)


Ridge Regression MSE: 50508878.307652056
Lasso Regression MSE: 50508855.78114253


##### 랜덤포레스트 - 동시학습 후 투표

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# 데이터 로드 및 전처리
df = pd.read_csv('datasets/Clean_Dataset.csv')
df = df.drop(['flight', 'departure_time', 'stops', 'arrival_time'], axis=1) # 학습에 필요 없는 문자열 열 제거
df = pd.get_dummies(df, columns=['airline', 'source_city', 'destination_city', 'class'], drop_first=True)
X = df.drop('price', axis=1)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 랜덤 포레스트 모델
rf = RandomForestRegressor(
    n_estimators=100,      # 트리 개수 (투표할 결정 트리의 수 - 많을 수록 안정적)
    max_depth=10,          # 각 트리의 최대 깊이 (깊을수록 복잡한 학습 / 과적합 위험)
    min_samples_split=5,   # 노드를 분할하기 위한 최소 샘플 수 (작을 수록 복잡한 모델)
    min_samples_leaf=2,    # 리프 노드에 있어야 하는 최소 샘플 수 (의결정족수)
    random_state=42        # 결과 재현성을 위한 설정
)
rf.fit(X_train, y_train)  # 모델 학습
rf_preds = rf.predict(X_test)  # 테스트 데이터 예측
rf_mse = mean_squared_error(y_test, rf_preds)  # MSE 계산

print("Random Forest MSE:", rf_mse)


Random Forest MSE: 20024154.727510292


##### 그래디언트 부스트 - 순차적인 학습

In [7]:
from sklearn.ensemble import GradientBoostingRegressor

# 데이터 로드 및 전처리
df = pd.read_csv('datasets/Clean_Dataset.csv')
df = df.drop(['flight', 'departure_time', 'stops', 'arrival_time'], axis=1) # 학습에 필요 없는 문자열 열 제거
df = pd.get_dummies(df, columns=['airline', 'source_city', 'destination_city', 'class'], drop_first=True)
X = df.drop('price', axis=1)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 그래디언트 부스트 모델
gb = GradientBoostingRegressor(
    learning_rate=0.1,     # 학습률 (랜덤포레스트 + 지속적인 학습 = 그래디언트 부스트) : 작을수록 신중한 학습 (일반화 좋음)
    n_estimators=100,      # 트리 개수
    max_depth=10,           # 각 트리의 최대 깊이
    random_state=42        # 결과 재현성을 위한 설정
)
gb.fit(X_train, y_train)  # 모델 학습
gb_preds = gb.predict(X_test)  # 테스트 데이터 예측
gb_mse = mean_squared_error(y_test, gb_preds)  # MSE 계산

print("Gradient Boosting MSE:", gb_mse)


KeyboardInterrupt: 